# Synthetic GT: Sinusoid + Olympic (merged)

Combines patterns from:
- `syntheticcreation sinusuid.ipynb` (sine / line strips)
- `syntheticcreation olympic.ipynb` (Olympic ring circles)

into **one** volume and **one** 3D visualization.


In [1]:
import numpy as np
from pathlib import Path

# Work relative to this notebook's folder
OUT_DIR = Path(".").resolve()
print("Working directory:", OUT_DIR)

# Shared dimensions (same as sinusoid / olympic notebooks)
num_channels = 4
value_range = (0, 10)
z_range = (0, 48)
y_range = (0, 343)
x_range = (0, 680)

num_values = value_range[1] - value_range[0] + 1
z_dim = z_range[1] - z_range[0] + 1
y_dim = y_range[1] - y_range[0] + 1
x_dim = x_range[1] - x_range[0] + 1

print(f"Volume shape: ({num_channels}, {num_values}, {z_dim}, {y_dim}, {x_dim})")


Working directory: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth
Volume shape: (4, 11, 49, 344, 681)


## 1) Build sinusoid strips + Olympic rings in one array


In [2]:
# ---------------------------------------------------------------------------
# Sinusoid helpers (from syntheticcreation sinusuid.ipynb)
# ---------------------------------------------------------------------------
def create_sine_strip(data, channel, y_center, strip_size, amplitude, period, z_dim, y_dim, x_dim):
    strip_half = strip_size // 2
    for z in range(z_dim):
        for x in range(x_dim):
            y_sine = y_center + amplitude * np.sin(2 * np.pi * x / period)
            y_sine_int = int(round(y_sine))
            y_start = max(0, y_sine_int - strip_half)
            y_end = min(y_dim, y_sine_int + strip_half + 1)
            for y in range(y_start, y_end):
                random_value = np.random.randint(0, 11)
                data[channel, random_value, z, y, x] = 1


def create_line_strip(data, channel, y_center, strip_size, z_dim, y_dim, x_dim):
    strip_half = strip_size // 2
    y_start = max(0, y_center - strip_half)
    y_end = min(y_dim, y_center + strip_half + 1)
    for z in range(z_dim):
        for x in range(x_dim):
            for y in range(y_start, y_end):
                random_value = np.random.randint(0, 11)
                data[channel, random_value, z, y, x] = 1


# ---------------------------------------------------------------------------
# Olympic helpers (from syntheticcreation olympic.ipynb)
# ---------------------------------------------------------------------------
def create_striped_circle(data, channel, center_x, center_y, max_radius, stripe_width, z_dim, y_dim, x_dim):
    inner_radius = max_radius - stripe_width
    outer_radius = max_radius
    for z in range(z_dim):
        for y in range(y_dim):
            for x in range(x_dim):
                dx = x - center_x
                dy = y - center_y
                distance = np.sqrt(dx**2 + dy**2)
                if inner_radius <= distance <= outer_radius:
                    random_value = np.random.randint(0, 11)
                    data[channel, random_value, z, y, x] = 1


# ---------------------------------------------------------------------------
# Create both patterns, then OR-merge presence
# ---------------------------------------------------------------------------
np.random.seed(42)

data_sinus = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)
data_olympic = np.zeros((num_channels, num_values, z_dim, y_dim, x_dim), dtype=np.int8)

# Sinusoid params
y_center = 100
strip_size = 20
sine_params = [
    (0, 100, 400),
    (1, 50, 200),
    (2, 150, 300),
]
for channel, amplitude, period in sine_params:
    create_sine_strip(data_sinus, channel, y_center, strip_size, amplitude, period, z_dim, y_dim, x_dim)
create_line_strip(data_sinus, 3, y_center, strip_size, z_dim, y_dim, x_dim)

# Olympic params
stripe_width = 30
circles = [
    (0, 200, 150, 120),
    (1, 420, 150, 120),
    (2, 300, 250, 120),
    (3, 350, 330, 200),
]
for channel, center_x, center_y, radius in circles:
    create_striped_circle(data_olympic, channel, center_x, center_y, radius, stripe_width, z_dim, y_dim, x_dim)

# Merge: keep markers from either pattern
data = np.maximum(data_sinus, data_olympic)

out_raw = OUT_DIR / "groundtruth_sinus_olympic.npy"
np.save(out_raw, data)
print(f"Saved merged 4-channel volume -> {out_raw.name}")
print(f"  shape: {data.shape}, dtype: {data.dtype}")
print(f"  sinus nonzeros:   {np.count_nonzero(data_sinus):,}")
print(f"  olympic nonzeros: {np.count_nonzero(data_olympic):,}")
print(f"  merged nonzeros:  {np.count_nonzero(data):,}")


Saved merged 4-channel volume -> groundtruth_sinus_olympic.npy
  shape: (4, 11, 49, 344, 681), dtype: int8
  sinus nonzeros:   2,618,756
  olympic nonzeros: 3,670,492
  merged nonzeros:  6,277,274


## 2) Add GT channel (voxels where 2+ channels overlap)


In [3]:
data = np.load(OUT_DIR / "groundtruth_sinus_olympic.npy")
n_channels, n_values, n_z, y_dim, x_dim = data.shape

channel_presence = np.sum(data, axis=1)
channel_binary = (channel_presence > 0).astype(np.int8)
channels_per_voxel = np.sum(channel_binary, axis=0)
intersection_mask = channels_per_voxel >= 2

z_coords, y_coords, x_coords = np.where(intersection_mask)

print(f"Loaded shape: {data.shape}")
for ch in range(n_channels):
    print(f"  Channel {ch} voxels: {np.sum(channel_binary[ch] > 0):,}")
print(f"Intersection voxels (2+ channels): {len(z_coords):,}")

ground_truth = np.zeros((n_channels + 1, n_values, n_z, y_dim, x_dim), dtype=data.dtype)
ground_truth[:n_channels] = data
gt_channel_idx = n_channels
gt_value_idx = 0
ground_truth[gt_channel_idx, gt_value_idx, z_coords, y_coords, x_coords] = 1

out_gt = OUT_DIR / "ground_truth_sinus_olympic.npy"
np.save(out_gt, ground_truth)
print(f"Saved 5-channel GT volume -> {out_gt.name}")
print(f"  Final shape: {ground_truth.shape}")


Loaded shape: (4, 11, 49, 344, 681)
  Channel 0 voxels: 1,599,213
  Channel 1 voxels: 1,624,105
  Channel 2 voxels: 1,338,876
  Channel 3 voxels: 1,595,048
Intersection voxels (2+ channels): 1,098,188
Saved 5-channel GT volume -> ground_truth_sinus_olympic.npy
  Final shape: (5, 11, 49, 344, 681)


## 3) One 3D visualization showing both patterns + GT


In [9]:
import os
import webbrowser
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import plotly.io as pio

# Self-contained: works even if only this cell is re-run
OUT_DIR = Path(".").resolve()
npy_path = OUT_DIR / "ground_truth_sinus_olympic.npy"
if not npy_path.exists():
    raise FileNotFoundError(
        f"Missing {npy_path.name}. Run the previous cells first "
        f"(working dir should be Groundtruth/). Current: {OUT_DIR}"
    )

# VS Code / Cursor / JupyterLab: mimetype works better than "notebook"
pio.renderers.default = "plotly_mimetype+notebook_connected"

data = np.load(npy_path)
n_channels, n_values, n_z, y_dim, x_dim = data.shape

z_range = (0, n_z - 1)
x_range = (0, x_dim - 1)
y_range = (0, y_dim - 1)

channel_colors = [
    "#a6cee3",  # 0
    "#1f78b4",  # 1
    "#b2df8a",  # 2
    "#33a02c",  # 3
    "#000000",  # 4 GT
]

per_channel_3d = np.zeros((n_channels, n_z, y_dim, x_dim), dtype=np.float32)

for ch in range(n_channels - 1):
    channel_data = data[ch]
    value_indices = np.arange(n_values, dtype=np.float32).reshape(-1, 1, 1, 1)
    weighted_sum = np.sum(value_indices * channel_data, axis=0)
    total_sum = np.sum(channel_data, axis=0)
    avg_value_index = np.zeros_like(weighted_sum)
    mask = total_sum > 0
    avg_value_index[mask] = weighted_sum[mask] / total_sum[mask]
    per_channel_3d[ch] = avg_value_index

gt_spots = data[n_channels - 1][0]
per_channel_3d[n_channels - 1] = gt_spots.astype(np.float32)

fig = go.Figure()
threshold = 0.5
max_points = 5000

for ch in range(n_channels - 1):
    vol = per_channel_3d[ch]
    if np.all(vol == 0):
        continue
    vmin, vmax = vol.min(), vol.max()
    if vmax <= vmin:
        continue
    norm = (vol - vmin) / (vmax - vmin + 1e-6)
    mask = norm > threshold
    if not np.any(mask):
        continue
    z_idx, y_idx, x_idx = np.where(mask)
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx, y_idx, z_idx = x_idx[selected], y_idx[selected], z_idx[selected]
    fig.add_trace(
        go.Scatter3d(
            x=x_idx,
            y=y_idx,
            z=z_idx,
            mode="markers",
            marker=dict(size=3.5, color=channel_colors[ch], opacity=0.55),
            name=f"Channel {ch}",
        )
    )

vol = per_channel_3d[n_channels - 1]
if np.any(vol > 0):
    z_idx, y_idx, x_idx = np.where(vol > 0)
    total_gt = len(z_idx)
    n_pts = len(x_idx)
    if n_pts > max_points:
        selected = np.random.choice(n_pts, max_points, replace=False)
        x_idx, y_idx, z_idx = x_idx[selected], y_idx[selected], z_idx[selected]
    fig.add_trace(
        go.Scatter3d(
            x=x_idx,
            y=y_idx,
            z=z_idx,
            mode="markers",
            marker=dict(size=3.5, color=channel_colors[n_channels - 1], opacity=0.4, line=dict(width=0)),
            name="GT spots",
        )
    )
    print(f"GT spots displayed: {len(z_idx)} / {total_gt} (max_points={max_points})")

x_span = x_range[1] - x_range[0] + 1
y_span = y_range[1] - y_range[0] + 1
z_span = z_range[1] - z_range[0] + 1
max_span = max(x_span, y_span, z_span)
aspect_x = x_span / max_span
aspect_y = y_span / max_span
aspect_z = z_span / max_span

fig.update_layout(
    title="3D Visualization: Sinusoid + Olympic (merged) + GT",
    scene=dict(
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
        xaxis=dict(range=[x_range[0], x_range[1]]),
        yaxis=dict(range=[y_range[0], y_range[1]]),
        zaxis=dict(range=[z_range[0], z_range[1]]),
        aspectmode="manual",
        aspectratio=dict(x=aspect_x, y=aspect_y, z=aspect_z),
        bgcolor="white",
    ),
    autosize=True,
    showlegend=True,
    legend=dict(
        itemsizing="constant",  # larger fixed legend markers (not tiny like data points)
        itemwidth=40,
        font=dict(size=16),
        bgcolor="rgba(255,255,255,0.85)",
        borderwidth=0,
    ),
)

print("Merged view: sinusoid strips + Olympic rings in one figure")

# Show inline; if that fails, open HTML in browser (same as olympic notebook)
html_file = OUT_DIR / "3d_visualization_sinus_olympic_gt.html"
try:
    fig.show()
except Exception as e:
    print(f"Inline show failed ({e}); writing HTML fallback...")
    fig.write_html(str(html_file))
    webbrowser.open("file:///" + os.path.abspath(html_file).replace("\\", "/"))
    print(f"Opened: {html_file}")
else:
    # Always also save HTML so you can open it outside the notebook
    fig.write_html(str(html_file))
    print(f"Also saved: {html_file}")


GT spots displayed: 5000 / 1098188 (max_points=5000)
Merged view: sinusoid strips + Olympic rings in one figure


Inline show failed (Mime type rendering requires nbformat>=4.2.0 but it is not installed); writing HTML fallback...
Opened: D:\VIS2025\BIoVisChallenges\SSGAT\Groundtruth\3d_visualization_sinus_olympic_gt.html
